### https://www.kaggle.com/competitions/drawing-with-llms

In [2]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
sys.path.append('./utils')

### Random seed for reproducibility

In [3]:
import torch
import random
import numpy as np
#import multiprocessing as mp
#mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [4]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import gc
import svg_constraints 
from svg_processor import SVGSanitizer, SVGProcessor

class Model:
    
    def __init__(self):

        self.model_path="./lora/Qwen3_4B_lora_fp16_r256_s60000_e2_1_msl2048"
        self.model = LLM(
            model=self.model_path,
            max_model_len=1024,
            #quantization="AWQ",
            gpu_memory_utilization=0.95,
            dtype="half",
            seed=123,
            disable_log_stats=True
        )
       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     


    def _format_prompt(self, description: str) -> str:
        return  f"""Below is an instruction that describes a task, paired with an input that provides further context. 
                Write a response that appropriately completes the request.
                
                ### Instruction:
                Generate a SVG code for the given input:
                
                ### Input:
                {description}
                
                ### Response:
                """
    
    def get_response(self, descriptions):
        
        formatted_input = [self._format_prompt(desc) for desc in descriptions]
        sampling_params = SamplingParams(temperature=0.9,top_k=50,  top_p=0.95, max_tokens=1024,n=1)
        outputs = self.model.generate(formatted_input, sampling_params)
        
        #suitable for batch inputs as well
        output_list=[]
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
            output_list.append(generated_text.strip())
        return output_list
    
    def predict(self, descriptions: list[str], max_new_tokens=1024) -> list[str]:
        output_decoded_list = self.get_response(descriptions)
        final_svg_code_list = []
    
        for description, output in zip(descriptions, output_decoded_list):
            base_svg = SVGProcessor.clean_and_extract_svgs(output, self.default_svg)
            clean_svg = self.sanitizer.enforce_constraints(base_svg)
            final_svg = SVGProcessor.svg_conversion_check(description, clean_svg, self.default_svg)
            
            #print('output:\n',output)
            #print('final_svg:\n',final_svg)
            
            final_svg_code_list.append(final_svg)
    
        return final_svg_code_list


INFO 05-17 09:35:02 [__init__.py:239] Automatically detected platform cuda.


In [5]:
model=Model()

WARNING 05-17 09:35:03 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 05-17 09:35:08 [config.py:585] This model supports multiple tasks: {'classify', 'score', 'reward', 'embed', 'generate'}. Defaulting to 'generate'.
INFO 05-17 09:35:08 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-17 09:35:09 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/Qwen3_4B_lora_fp16_r256_s60000_e2_1_msl2048', speculative_config=None, tokenizer='./lora/Qwen3_4B_lora_fp16_r256_s60000_e2_1_msl2048', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_deco

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 05-17 09:35:13 [loader.py:447] Loading weights took 2.68 seconds
INFO 05-17 09:35:13 [gpu_model_runner.py:1186] Model loading took 7.5454 GB and 2.981297 seconds
INFO 05-17 09:35:14 [kv_cache_utils.py:566] GPU KV cache size: 7,968 tokens
INFO 05-17 09:35:14 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 7.78x
INFO 05-17 09:35:19 [gpu_model_runner.py:1534] Graph capturing finished in 5 secs, took 0.09 GiB
INFO 05-17 09:35:20 [core.py:151] init engine (profile, create kv cache, warmup model) took 6.77 seconds


In [6]:
import sys
sys.path.append(r'/home/vino/ML_Projects/Drawing_with_LLMs/drawing-with-llms')
import pandas as pd

df1=pd.read_csv(r'./drawing-with-llms/test_filtered_1_batch_vqa_gpt4.csv',header=[0])
df2=pd.read_csv(r'./drawing-with-llms/description_master_test_gemini_25pro_2k.csv',header=[0])
#df3=pd.read_csv(r'./drawing-with-llms/gemini_25_pro_validation/train_filtered_1_batch_gpt4.csv',header=[0])
#print(df3.shape)
#df2=df2.drop_duplicates(['description'])
#df=pd.concat([df1[['description']],df2['description']],axis=0)
df=df1.drop_duplicates(['description'])
#df=df.iloc[:300]
print(df.shape)
df.head(2)

(71, 7)


,description,clean_svg,sl_score,response,vqa_pair,response_2,gpt_svg_2
0,Vibrant autumn forest,"<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.904117,Here is the visual question answering (VQA) pa...,"{'description': 'Vibrant autumn forest', 'ques...","Here's an improved SVG representation of a ""Vi...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,Morning dew on grass,"<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.987024,Here is a visual question answering (VQA) pair...,"{'description': 'Morning dew on grass', 'quest...","Here's an improved SVG representation of ""Morn...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [11]:
#_= model.predict([df['description'].iloc[3]])

Processed prompts: 100%|█| 1/1 [00:18<00:00, 18.93s/it, est. speed input: 3.12 t

output:
 <svg width="200" height="150" viewBox="0 0 200 150">
  <defs>
    <linearGradient id="skyGradient" x1="0%" y1="0%" x2="0%" y2="100%">
      <stop offset="0%" stop-color="#87CEEB" />
      <stop offset="100%" stop-color="#ADD8E6" />
    </linearGradient>
  </defs>
  <rect x="0" y="0" width="200" height="100" fill="url(#skyGradient)" />
  <path d="M0 100 C0 100, 100 120, 200 100 L200 150 L0 150 Z" fill="#F2E6B5" />
  <path d="M10 110 C10 110, 20 120, 30 110" fill="#66CDAA" stroke="#556B2F" stroke-width="2"/>
  <path d="M40 110 C40 110, 50 120, 60 110" fill="#66CDAA" stroke="#556B2F" stroke-width="2"/>
  <path d="M70 110 C70 110, 80 120, 90 110" fill="#66CDAA" stroke="#556B2F" stroke-width="2"/>
  <circle cx="100" cy="130" r="10" fill="#FFD700" />
  <circle cx="120" cy="135" r="8" fill="#FFD700" />
  <circle cx="140" cy="125" r="6" fill="#FFD700" />
  <line x1="10" y1="100" x2="10" y2="110" stroke="#000080" stroke-width="2" stroke-linecap="round"/>
  <line x1="190" y1="100" x2="1

In [ ]:
# from tqdm import tqdm
# tqdm.pandas()
# df['svg_3'] = df['description'].progress_apply(lambda x: model.predict(x))

In [ ]:
from tqdm import tqdm
description_list = [s.strip(" ',") for s in df['description'].to_list()]
batch_size = 15
results = []

for i in tqdm(range(0, len(description_list), batch_size), desc="Batch prediction"):
    batch = description_list[i:i + batch_size]
    batch_result = model.predict(batch)  # Ensure this handles a list of inputs
    results.extend(batch_result)


In [ ]:
df['svg_3']=results

In [ ]:
model.close_model()

In [ ]:
from siglip_class import SVGMetricEvaluator
from aesthetic_evaluator import AestheticEvaluator

In [ ]:
#SigLip Score
from tqdm import tqdm
tqdm.pandas()
evaluator = SVGMetricEvaluator()
df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)

In [ ]:
#Aes Score
from tqdm import tqdm
tqdm.pandas()
aes_eval = AestheticEvaluator()
df['aes_score_3'] = df.progress_apply(lambda row: aes_eval.get_score(row['svg_3']), axis=1)

In [ ]:
#combined score
df['combined_score_3'] = (df['svg_score_3']+df['svg_score_3']+df['aes_score_3'])/3

In [ ]:
print('mean_svg_score:',df['svg_score_3'].mean(),'mean_aes_score:',df['aes_score_3'].mean(),'combined_score:',df['combined_score_3'].mean())

In [ ]:
default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
df_default_svg=df[df['svg_3']==default_svg]
print('default_svg_count:',df_default_svg.shape[0])
print('default_svg_score_mean:',df_default_svg['svg_score_3'].mean(),'default_aes_score_mean:',df_default_svg['aes_score_3'].mean(),\
     'combined_score:',df_default_svg['combined_score_3'].mean())

In [ ]:
df_non_default_svg=df[df['svg_3']!=default_svg]
print('non-default_svg_count:',df_non_default_svg.shape[0])
print('non-default_svg_score_mean:',df_non_default_svg['svg_score_3'].mean(),\
      'non-default_aes_score_mean:',df_non_default_svg['aes_score_3'].mean(),\
        'combined_score:',df_non_default_svg['combined_score_3'].mean())

In [ ]:
df['svg_3'].iloc[2]